# 🧬 SynapseMemory: Developer Walkthrough & API Notebook

Welcome to the official developer playbook for **SynapseMemory**—the Universal Active RAG & Cognitive Memory Layer.

In this notebook, we will exercise the complete Phase 1 MVP algorithmic pipeline, demonstrating:
1. Creating a durable, local relational **SQLite storage backend**.
2. Generating vector embeddings using the **local deterministic DJB2 vectorizer**.
3. Running **Semantic Deduplication** to prevent prompt-context fragmentation.
4. Calculating **Ebbinghaus exponential temporal decay** weights.
5. Solving prompt context budget limits using the **0/1 Knapsack Dynamic Programmer**.

### Step 1: Initialize the Engine and Database Store

First, we set up our core imports and initialize a clean local SQLite store and local embedder module.

In [ ]:
import sys
import os
import time

# Ensure PYTHONPATH is aligned
sys.path.append(os.path.abspath('..'))

from synapse_memory.core.sqlite_db import SQLiteMemoryStore
from synapse_memory.core.embedder import SynapseEmbedder
from synapse_memory.core.memory_manager import MemoryManager
from synapse_memory.core.knapsack_packer import KnapsackPacker
from synapse_memory.core.decay_engine import DecayEngine
from synapse_memory.core.hybrid_search import HybridSearch

# Setup self-initializing SQLite relational database
store = SQLiteMemoryStore("synapse_memory_sandbox.db")
embedder = SynapseEmbedder(provider="local")
manager = MemoryManager(store=store, embedder=embedder, max_record_limit=5)

print(f"[+] SQLite Memory Store initialized. Total nodes in sandbox: {store.count_memories()}")

### Step 2: Demonstrate Semantic Deduplication & Anti-Fragmentation

Let's ingest two highly similar texts. Instead of creating two separate vector nodes and wasting precious context-window tokens, the deduplication engine will automatically identify the semantic match and merge/reinforce the existing record.

In [ ]:
sentence1 = "Developers must configure secure SSL certificates on corporate firewalls."
sentence2 = "DEVELOPERS MUST CONFIGURE SECURE SSL CERTIFICATES ON CORPORATE FIREWALLS!!!"

# First ingestion
id1, action1, cost1 = manager.ingest_with_deduplication(sentence1, "security")
print(f"Ingestion 1 Status: Node {id1} was {action1} (Cost: {cost1} tokens)")

# Second ingestion (Triggers deduplication merge loop)
id2, action2, cost2 = manager.ingest_with_deduplication(sentence2, "security")
print(f"Ingestion 2 Status: Node {id2} was {action2} (Cost: {cost2} tokens)")

print(f"Total unique memories stored in SQLite database: {store.count_memories()}")
assert id1 == id2, "Deduplication failed to map identical nodes!"

### Step 3: Run Ingestion across Diverse Topics

Let's ingest distinct technical contexts to populate our SQLite long-term memory layer.

In [ ]:
contexts = [
    "SpaceX launched a high-orbit telemetry rocket from Cape Canaveral.",
    "Nginx reverse proxy servers bind exclusively to port 3000 inside containers.",
    "Apple pie dessert recipes require flour, fresh cinnamon, and sugar.",
    "Docker organizes lightweight container hypervisors cleanly across Linux nodes."
]

for ctx in contexts:
    mem_id, action, cost = manager.ingest_with_deduplication(ctx, "kb")
    print(f"Indexed Node {mem_id}: {action} (Cost: {cost} tokens)")

print(f"Total nodes indexed in SQLite: {store.count_memories()}")

### Step 4: Hybrid Query, Temporal Decay & Knapsack Prompt Packing

We query our long-term memory for relevant data, calculate Ebbinghaus temporal decay retention scores, and solve context budgets using 0/1 Dynamic Programming Knapsack constraints.

In [ ]:
query = "What container port and proxy settings should we deploy on Linux server?"
print(f"User Prompt: '{query}'\n")

# 1. Hybrid semantic search
all_mems = store.get_all_memories()
searcher = HybridSearch()
candidates = searcher.fused_search(query, all_mems, top_k=5)

# 2. Calculate Ebbinghaus temporal decay retention weights
decay_eng = DecayEngine()
decayed_candidates = []
for c in candidates:
    ret_score = decay_eng.calculate_retention(
        base_relevance=c["relevance_score"],
        created_epoch=c["created_at"],
        access_count=c["access_count"]
    )
    c_copy = c.copy()
    c_copy["relevance_score"] = ret_score
    decayed_candidates.append(c_copy)
    print(f"Node [{c['id']}]: Cosine Similarity={c['relevance_score']:.3f} | Temporal Weight={ret_score:.3f}")

# 3. Pack optimal context subset using 0/1 Dynamic Programming Knapsack
packer = KnapsackPacker()
token_budget = 80 # tight token constraint
packed_result = packer.pack(decayed_candidates, token_budget)

print(f"\n[+] Packed Context Subset (Total Cost: {sum(m['token_cost'] for m in packed_result)}/{token_budget} tokens):")
for idx, node in enumerate(packed_result):
    print(f"  {idx + 1}. [{node['id']}] Category: {node['category'].upper()} (Cost: {node['token_cost']} tokens):\n     \"{node['content']}\"")

### Cleanup Sandbox Database

We clean up the sandbox database file to restore original workspace boundaries.

In [ ]:
if os.path.exists("synapse_memory_sandbox.db"):
    os.remove("synapse_memory_sandbox.db")
    print("[+] Cleaned up sandbox relational SQLite tables successfully!")